# 🎬 Video Subtitle Remover

**Remove hardcoded subtitles from videos using AI**

## How to use:
1. Click **Runtime** → **Run all** (or press Ctrl+F9)
2. Wait for installation (2-3 minutes)
3. Click the **public URL** that appears at the bottom
4. Upload your video and remove subtitles!

---

In [ ]:
#@title 1️⃣ Install Dependencies (Run this first)
!pip install -q gradio easyocr opencv-python-headless numpy torch torchvision

In [ ]:
#@title 2️⃣ Launch App (Run this second - Get your link below!)

import gradio as gr
import cv2
import numpy as np
import tempfile
import easyocr

print("Loading AI model... (this takes 1-2 minutes first time)")
reader = easyocr.Reader(['en'], gpu=True, verbose=False)
print("✅ Model loaded!")

def process_video(video_file, subtitle_region, progress=gr.Progress()):
    if video_file is None:
        return None, "❌ Please upload a video."

    progress(0.1, desc="Opening video...")
    cap = cv2.VideoCapture(video_file)

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    output_path = tempfile.mktemp(suffix='.mp4')
    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    subtitle_y = int(height * (1 - subtitle_region))
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        roi = frame[subtitle_y:, :]
        results = reader.readtext(roi)

        mask = np.zeros((height, width), dtype=np.uint8)
        for bbox, text, conf in results:
            if conf > 0.3:
                pts = np.array(bbox, dtype=np.int32)
                pts[:, 1] += subtitle_y
                x1, x2 = max(0, int(np.min(pts[:,0]))-10), min(width, int(np.max(pts[:,0]))+10)
                y1, y2 = max(0, int(np.min(pts[:,1]))-10), min(height, int(np.max(pts[:,1]))+10)
                mask[y1:y2, x1:x2] = 255

        if np.sum(mask) > 0:
            frame = cv2.inpaint(frame, mask, 7, cv2.INPAINT_TELEA)

        writer.write(frame)
        frame_count += 1

        if frame_count % 15 == 0:
            progress(0.1 + (frame_count/total_frames)*0.85, desc=f"Frame {frame_count}/{total_frames}")

    cap.release()
    writer.release()

    return output_path, f"✅ Done! Processed {frame_count} frames."

# Create UI
with gr.Blocks(title="Video Subtitle Remover", theme=gr.themes.Soft()) as demo:
    gr.HTML("<h1 style='text-align:center'>🎬 Video Subtitle Remover</h1>")
    gr.HTML("<p style='text-align:center'>Upload a video to remove hardcoded subtitles</p>")

    with gr.Row():
        with gr.Column():
            video_in = gr.Video(label="Upload Video")
            region = gr.Slider(0.15, 0.5, 0.35, step=0.05, label="Subtitle Region (bottom %)")
            btn = gr.Button("🚀 Remove Subtitles", variant="primary")

        with gr.Column():
            video_out = gr.Video(label="Clean Video")
            status = gr.Markdown()

    btn.click(process_video, [video_in, region], [video_out, status])

print("\n" + "="*50)
print("🚀 LAUNCHING APP...")
print("="*50)
demo.launch(share=True)